# APCS — example usage

The **Arabic-Aware Prompt Compression Selector (APCS)** recommends, for an Arabic prompt, a
prompt-compression method and target keep rate. Three frozen selectors ship with the package:

| Selector | Needs | Optimises |
|---|---|---|
| `v1` (APCS 1.0.0, default) | prompt | best balance: most tokens removed while output fidelity stays ≥ τ |
| `v2` (APCS-v2) | prompt + task category | best balance (most accurate on the fresh exam) |
| `cost` (APCS-cost) | prompt + task category | cheapest API call with output fidelity ≥ τ |

Install from the repository root: `pip install -e apcs` (add `[compression]` to apply recommendations).

In [1]:
import json
from pathlib import Path
import pandas as pd
from apcs import APCSSelector, extract_features

v1, v2, cost = APCSSelector("v1"), APCSSelector("v2"), APCSSelector("cost")

## 1. A short prompt: no compression
Short prompts are left alone; every recommendation states the rule that produced it.

In [2]:
rec = v1.recommend("اشرح مفهوم الذكاء الاصطناعي بأسلوب مبسط.")
print(rec.method, rec.rate, "|", rec.rule)

none 1.0 | token_count 32 < 80


## 2. One prompt, three selectors
A summarisation prompt from the AraPromptBench v2 dev split that contains a length instruction ("in one paragraph").

In [3]:
bench = json.loads(Path("../../AraPromptBench_v2.json").read_text(encoding="utf-8"))["prompts"]
prompt = next(p for p in bench if p["id"] == "v2-summ-051")
print(prompt["prompt"][:200], "...")
for name, sel in [("v1", v1), ("v2", v2), ("cost", cost)]:
    r = sel.recommend(prompt["prompt"], category=prompt["category"]) if name != "v1" else sel.recommend(prompt["prompt"])
    print(f"{name:5s} -> {r.method} @ {r.rate}   rule: {r.rule}")

اكتب ملخصًا في فقرة واحدة يوضح الأفكار الرئيسية للنص الآتي:

يسعى الرئيس الإيراني السابق هاشمي رفسنجاني عبر ترشحه إلى الانتخابات الرئاسية
في إيران، إلى العودة بقوة إلى الساحتين الإيرانية والدولية ويطر ...
v1    -> llmlingua2 @ 0.5   rule: 80 <= token_count 191 < 250
v2    -> llmlingua2 @ 0.5   rule: cat_creative <= 0.5 and token_count > 104.5 and cat_qa <= 0.5 and token_count <= 243
cost  -> llmlingua2_protected @ 0.3   rule: token_count > 127.5 and has_length_instruction > 0.5 and cat_creative <= 0.5


The cost-aware selector keeps the length instruction (protected variant), because deleting it makes answers longer and the API bill higher.

In [4]:
f = extract_features(prompt["prompt"])
pd.Series(f.as_dict())

character_length             271
token_count                  191
word_count                    41
fragmentation_ratio       4.6585
structural_complexity          5
morphological_density       None
has_length_instruction      True
dtype: object

## 3. Recommendations for a batch of prompts

In [5]:
sample = [p for p in bench if p["split"] == "dev"][:200:25]
rows = []
for p in sample:
    rows.append({"id": p["id"], "category": p["category"], "tokens": p["token_count"],
                 "v1": "{0.method}@{0.rate}".format(v1.recommend(p["prompt"])),
                 "v2": "{0.method}@{0.rate}".format(v2.recommend(p["prompt"], category=p["category"])),
                 "cost": "{0.method}@{0.rate}".format(cost.recommend(p["prompt"], category=p["category"]))})
pd.DataFrame(rows)

,id,category,tokens,v1,v2,cost
0,v2-inst-071,instruction,199,llmlingua2@0.5,llmlingua2@0.5,none@1.0
1,v2-inst-096,instruction,32,none@1.0,llmlingua2@0.5,none@1.0
2,v2-inst-121,instruction,125,llmlingua2@0.5,llmlingua2@0.5,none@1.0
3,v2-inst-146,instruction,240,llmlingua2@0.5,llmlingua2@0.5,none@1.0
4,v2-inst-171,instruction,120,llmlingua2@0.5,llmlingua2@0.5,none@1.0
5,v2-inst-196,instruction,51,none@1.0,none@1.0,none@1.0
6,v2-inst-221,instruction,139,llmlingua2@0.5,llmlingua2@0.5,none@1.0
7,v2-inst-246,instruction,54,none@1.0,llmlingua2@0.5,none@1.0


## 4. Applying a recommendation
`compress()` recommends and then compresses with LLMLingua-2 (needs the `compression` extra; about 30 ms per prompt on a GPU, 0.4 s on CPU).

In [6]:
import tiktoken
enc = tiktoken.get_encoding("cl100k_base")
compressed = cost.compress(prompt["prompt"], category=prompt["category"])
print(len(enc.encode(prompt["prompt"])), "->", len(enc.encode(compressed)), "tokens")
print(compressed)

191 -> 90 tokens
اكتب ملخصًا في فقرة واحدة يوضح الأفكار الرئيسية للنص الآتي: هاشمي رفسنجاني الانتخابات العودة لإصلاحات اقتصادية العلاقات الدولي


## 5. Custom calibration
Rules are data: recalibrating for another LLM or price ratio means writing a new JSON file, not changing code.

In [7]:
import tempfile
custom = {"version": "example", "fidelity_threshold_tau": 0.65,
          "thresholds": {"T1": 120, "T2": 300, "r_mid": 0.7, "r_long": 0.5}}
path = Path(tempfile.mkdtemp()) / "rules.json"
path.write_text(json.dumps(custom))
print(APCSSelector("v1", rules_path=path).recommend(prompt["prompt"]).rule)

120 <= token_count 191 < 300
